In [2]:
import sys, os, subprocess, threading
sys.path.append('backend/app/')
import xarray as xr, numpy as np
from backend.app.services import functions
from backend.app.config import PROJECT_DES, DELFT_PATH

In [2]:
# Stream logs
def stream_logs(process, input_dir):
    try:
        for line in process.stdout:
            line = line.strip()
            if not line: continue
            print(line)
    except Exception as e:
        print(f"[INTERNAL ERROR] {e}")
    finally:
        process.wait()
        print("Reorganizing outputs. Please wait...")
        try:
            post_result = functions.postProcess(input_dir)
            print(post_result["message"])
        except Exception as e:
            print(f"Simulation failed: {e}")

In [ ]:
project = '3months'
dir = os.path.join(PROJECT_DES, f"backend/projects/vlnguyen/{project}")
input_dir = os.path.normpath(os.path.join(dir, "input"))
mdu_path = os.path.normpath(os.path.join(input_dir, "FlowFM.mdu"))
bat_path = os.path.normpath(os.path.join(DELFT_PATH, "x64/dflowfm/scripts/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2026_02/bin/run_dflowfm.bat"))
# bat_path = os.path.normpath(os.path.join(DELFT_PATH, "install_fm-suite_2025_01/bin/run_dflowfm.bat"))
command = ["cmd.exe", "/c", bat_path, "--autostartstop", mdu_path]

In [ ]:
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding="utf-8", errors="replace", bufsize=1, cwd=input_dir)
threading.Thread(target=stream_logs, args=(process, input_dir), daemon=True).start()

In [30]:
path = r"backend\projects\vlnguyen\demo\grids\Grid.nc"
ds = xr.open_dataset(path)
ds

<xarray.Dataset> Size: 457kB
Dimensions:            (mesh2d_nEdges: 4133, Two: 2, mesh2d_nFaces: 2559,
                        mesh2d_nMax_face_nodes: 4, mesh2d_nNodes: 1575)
Coordinates:
    mesh2d_node_x      (mesh2d_nNodes) float64 13kB ...
    mesh2d_node_y      (mesh2d_nNodes) float64 13kB ...
Dimensions without coordinates: mesh2d_nEdges, Two, mesh2d_nFaces,
                                mesh2d_nMax_face_nodes, mesh2d_nNodes
Data variables:
    crs                int32 4B ...
    mesh2d_edge_x      (mesh2d_nEdges) float64 33kB ...
    mesh2d_edge_y      (mesh2d_nEdges) float64 33kB ...
    mesh2d_edge_nodes  (mesh2d_nEdges, Two) int32 33kB ...
    mesh2d_face_nodes  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float64 82kB ...
    mesh2d_edge_faces  (mesh2d_nEdges, Two) int32 33kB ...
    mesh2d_face_x      (mesh2d_nFaces) float64 20kB ...
    mesh2d_face_y      (mesh2d_nFaces) float64 20kB ...
    mesh2d_face_x_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float64 82kB ...
    mesh2d_face_y_bnd  (mesh2d_nFaces, mesh2d_nMax_face_nodes) float64 82kB ...
    mesh2d             int32 4B ...
    mesh2d_node_z      (mesh2d_nNodes) float64 13kB ...
Attributes:
    institution:  Private
    references:   vanlnNTNU@gmail.com

In [31]:
ds['mesh2d_node_z'].values

array([0., 0., 0., ..., 0., 0., 0.])